# Gomoku 15×15: Policy CNN (học có giám sát)

**Bài toán:** đầu vào là bàn cờ (nhìn từ phía người sắp đi), nhãn là nước đi tiếp theo. Đây là bài toán **phân loại 225 lớp**.
Dữ liệu: các ván Gomocup 2019 Freestyle, lưu trong file `gomoku_15x15.npz` do `create_dataset.py` tạo ra.

**Chạy trên Kaggle**
1. Upload `gomoku_15x15.npz` lên thành một Kaggle Dataset (*Add Data → Upload*), rồi gắn dataset đó vào notebook.
2. Vào *Settings → Accelerator* và chọn *GPU T4 x1* (hoặc P100).
3. Bấm *Run All*. Model tốt nhất được lưu vào `/kaggle/working/gomoku_policy.pt`.

In [ ]:
import glob, os, time, hashlib
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device, torch.cuda.get_device_name(0) if device.type == 'cuda' else '')

N = 15
QUICK = os.environ.get('QUICK') == '1'   # smoke test on a tiny subset (local only)

## 1. Tải dữ liệu

In [ ]:
# Đổi thành 'gomoku_15x15_winner.npz' nếu chỉ muốn học từ nước đi của người thắng.
DATA_NAME = 'gomoku_15x15.npz'
cands = glob.glob(f'/kaggle/input/**/{DATA_NAME}', recursive=True) + glob.glob(f'data/{DATA_NAME}')
assert cands, f'Không tìm thấy {DATA_NAME}: hãy gắn Kaggle Dataset chứa file này vào notebook.'
d = np.load(cands[0])
X, y, game = d['X'], d['y'].astype(np.int64), d['game']
if QUICK:
    keep = game < 300
    X, y, game = X[keep], y[keep], game[keep]
print(cands[0], X.shape, y.shape, 'games:', game.max() + 1)

## 2. Chia train / val theo **ván** (không theo từng mẫu)
Các nước đi trong cùng một ván rất giống nhau. Nếu chia ngẫu nhiên theo từng mẫu, val sẽ chứa những thế cờ gần như đã có trong train, nên độ chính xác đo được sẽ cao hơn thực tế.
Ngoài ra nhiều ván dùng chung khai cuộc nên có thế cờ trùng nhau. Vì vậy ta bỏ khỏi val những thế cờ đã xuất hiện trong train.

In [ ]:
rng = np.random.default_rng(0)
games = np.unique(game)
val_games = rng.choice(games, size=len(games) // 10, replace=False)
is_val = np.isin(game, val_games)

pos_hash = np.array([hashlib.md5(b.tobytes()).digest() for b in X])
train_pos = set(pos_hash[~is_val])
leak = is_val & np.array([h in train_pos for h in pos_hash])
print(f'Bỏ {leak.sum()} / {is_val.sum()} mẫu val vì trùng thế cờ với train')

tr_idx = np.where(~is_val)[0]
va_idx = np.where(is_val & ~leak)[0]
print('train', len(tr_idx), 'val', len(va_idx))

Xem thử một mẫu: **X** là quân của người sắp đi, **O** là quân đối thủ, khung đỏ là nước đi thật (nhãn).

In [ ]:
def show(board2, move=None, probs=None, ax=None, title=''):
    ax = ax or plt.gca()
    ax.imshow(np.zeros((N, N)) if probs is None else probs.reshape(N, N), cmap='Blues', vmin=0)
    ax.set_xticks(range(N)); ax.set_yticks(range(N))
    for r in range(N):
        for c in range(N):
            if board2[0, r, c]: ax.text(c, r, 'X', ha='center', va='center', fontsize=12, weight='bold')
            if board2[1, r, c]: ax.text(c, r, 'O', ha='center', va='center', fontsize=12, color='gray', weight='bold')
    if move is not None:
        r, c = divmod(int(move), N)
        ax.add_patch(plt.Rectangle((c - .5, r - .5), 1, 1, fill=False, ec='red', lw=2))
    ax.set_title(title)

i = tr_idx[1234 % len(tr_idx)]
plt.figure(figsize=(5, 5)); show(X[i], y[i], title=f'label (row, col) = {divmod(int(y[i]), N)}'); plt.show()

## 3. Đưa toàn bộ dữ liệu lên GPU và tăng cường bằng 8 phép đối xứng
Bàn cờ có 8 phép đối xứng (4 góc xoay, mỗi góc có thể lật thêm). Mỗi batch áp một phép ngẫu nhiên cho cả bàn cờ lẫn nhãn.
Nhờ vậy dữ liệu hiệu dụng tăng gấp 8 lần mà không tốn thêm bộ nhớ.

In [ ]:
Xg = torch.from_numpy(X).to(device)          # uint8 (M, 2, 15, 15), ~150 MB
yg = torch.from_numpy(y).to(device)
tr_t = torch.from_numpy(tr_idx).to(device)
va_t = torch.from_numpy(va_idx).to(device)

def sym(t, k):
    # k in 0..7: rotate k%4 times, flip if k>=4. Works on (..., 15, 15).
    t = torch.rot90(t, k % 4, dims=(-2, -1))
    return torch.flip(t, dims=(-1,)) if k >= 4 else t

# label remap: inv[k][old_index] = new_index after symmetry k
grid = torch.arange(N * N, device=device).view(N, N)
inv = torch.empty(8, N * N, dtype=torch.long, device=device)
for k in range(8):
    inv[k, sym(grid, k).flatten()] = torch.arange(N * N, device=device)

def batches(idx, bs, train):
    if train:
        idx = idx[torch.randperm(len(idx), device=device)]
    for s in range(0, len(idx), bs):
        b = idx[s:s + bs]
        xb, yb = Xg[b], yg[b]
        if train:
            k = int(torch.randint(8, ()))
            xb, yb = sym(xb, k), inv[k][yb]
        yield xb, yb

# Sanity check: mark the label square on the board, transform, and the mark must land on the new label.
xb, yb = Xg[tr_t[:64]], yg[tr_t[:64]]
marked = torch.zeros(len(yb), N * N, device=device).scatter(1, yb[:, None], 1.).view(-1, N, N)
for k in range(8):
    assert torch.equal(sym(marked, k).flatten(1).argmax(1), inv[k][yb])
    assert (sym(xb, k).sum(1).flatten(1).gather(1, inv[k][yb][:, None]) == 0).all()
print('symmetry OK')

## 4. Mô hình: ResNet nhỏ, giữ nguyên kích thước 15×15 đến tận lớp cuối
- Đầu vào có 3 kênh: quân mình, quân đối thủ, và một kênh toàn số 1 (giúp mạng nhận ra mép bàn khi dùng zero-padding).
- Đầu ra: conv 1×1 cho ra 225 logit. Các ô đã có quân bị che (mask) trước softmax.

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c1 = nn.Conv2d(c, c, 3, padding=1, bias=False); self.b1 = nn.BatchNorm2d(c)
        self.c2 = nn.Conv2d(c, c, 3, padding=1, bias=False); self.b2 = nn.BatchNorm2d(c)
    def forward(self, x):
        h = F.relu(self.b1(self.c1(x)))
        return F.relu(x + self.b2(self.c2(h)))

class PolicyNet(nn.Module):
    def __init__(self, channels=96, blocks=8):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, channels, 5, padding=2, bias=False), nn.BatchNorm2d(channels), nn.ReLU())
        self.body = nn.Sequential(*[ResBlock(channels) for _ in range(blocks)])
        self.head = nn.Sequential(nn.Conv2d(channels, 32, 1, bias=False), nn.BatchNorm2d(32), nn.ReLU(),
                                  nn.Conv2d(32, 1, 1))
    def forward(self, board):                      # board: uint8 (B, 2, 15, 15)
        x = board.float()
        x = torch.cat([x, torch.ones_like(x[:, :1])], 1)
        logits = self.head(self.body(self.stem(x))).flatten(1).float()
        occupied = (board[:, 0] + board[:, 1]).flatten(1) > 0
        return logits.masked_fill(occupied, -1e9)

CFG = dict(channels=96, blocks=8)
model = PolicyNet(**CFG).to(device)
print(f'{sum(p.numel() for p in model.parameters()) / 1e6:.2f}M params')

## 5. Huấn luyện

In [ ]:
EPOCHS = 1 if QUICK else 20
BS = 512
opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
steps = EPOCHS * ((len(tr_idx) + BS - 1) // BS)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2e-3, total_steps=steps, pct_start=0.1)
use_amp = device.type == 'cuda'
scaler = torch.amp.GradScaler(enabled=use_amp)

@torch.no_grad()
def evaluate():
    model.eval()
    loss = top1 = top5 = 0
    for xb, yb in batches(va_t, 2048, train=False):
        with torch.autocast(device.type, dtype=torch.float16, enabled=use_amp):
            out = model(xb)
        loss += F.cross_entropy(out, yb, reduction='sum').item()
        t5 = out.topk(5, 1).indices
        top1 += (t5[:, 0] == yb).sum().item()
        top5 += (t5 == yb[:, None]).any(1).sum().item()
    n = len(va_idx)
    return loss / n, top1 / n, top5 / n

hist, best = [], 0.0
for ep in range(EPOCHS):
    model.train(); t0 = time.time(); tl = tn = 0
    for xb, yb in batches(tr_t, BS, train=True):
        with torch.autocast(device.type, dtype=torch.float16, enabled=use_amp):
            loss = F.cross_entropy(model(xb), yb)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(opt); scaler.update(); sched.step()
        tl += loss.item() * len(yb); tn += len(yb)
    vl, v1, v5 = evaluate()
    hist.append((tl / tn, vl, v1, v5))
    if v1 > best:
        best = v1
        torch.save({'cfg': CFG, 'state_dict': model.state_dict()}, 'gomoku_policy.pt')
    print(f'ep {ep+1:2d}/{EPOCHS}  train loss {tl/tn:.3f}  val loss {vl:.3f}  top1 {v1:.3f}  top5 {v5:.3f}  ({time.time()-t0:.0f}s)')
print('best val top1', best)

In [ ]:
h = np.array(hist)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(h[:, 0], label='train'); ax[0].plot(h[:, 1], label='val'); ax[0].set_title('loss'); ax[0].legend()
ax[1].plot(h[:, 2], label='top1'); ax[1].plot(h[:, 3], label='top5'); ax[1].set_title('val accuracy'); ax[1].legend()
plt.show()

## 6. Xem mô hình dự đoán
Màu xanh thể hiện xác suất mô hình gán cho từng ô, khung đỏ là nước đi thật của engine.

In [ ]:
ckpt = torch.load('gomoku_policy.pt', map_location=device)
model.load_state_dict(ckpt['state_dict']); model.eval()

fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
for ax, i in zip(axes, rng.choice(va_idx, 3, replace=False)):
    with torch.no_grad():
        p = model(Xg[i:i + 1]).softmax(1)[0].cpu().numpy()
    pred = p.argmax()
    show(X[i], y[i], probs=p, ax=ax, title=f'pred {divmod(int(pred), N)}  true {divmod(int(y[i]), N)}  p={p[y[i]]:.2f}')
plt.show()

## 7. Dùng mô hình để chơi
Hàm `best_move(board, player)` nhận `board` là mảng 15×15 (0 = trống, 1 = đen, 2 = trắng) và `player` ∈ {1, 2} là người sắp đi.
Bên dưới là một ván tự đấu (model đấu với chính nó), bắt đầu từ một quân ở giữa bàn. Nếu ván cờ trông hợp lý thì mô hình đã học được điều gì đó.

In [ ]:
@torch.no_grad()
def best_move(board, player):
    planes = np.stack([board == player, board == 3 - player]).astype(np.uint8)
    logits = model(torch.from_numpy(planes)[None].to(device))[0]
    return divmod(int(logits.argmax()), N)          # (row, col)

def five(board, r, c):
    p = board[r, c]
    for dr, dc in ((0, 1), (1, 0), (1, 1), (1, -1)):
        n = 1
        for s in (1, -1):
            rr, cc = r + s * dr, c + s * dc
            while 0 <= rr < N and 0 <= cc < N and board[rr, cc] == p:
                n += 1; rr += s * dr; cc += s * dc
        if n >= 5: return True
    return False

board = np.zeros((N, N), np.int8); board[7, 7] = 1
player, winner = 2, 0
for _ in range(N * N - 1):
    r, c = best_move(board, player)
    board[r, c] = player
    if five(board, r, c): winner = player; break
    player = 3 - player
print('winner:', {0: 'draw', 1: 'black (X)', 2: 'white (O)'}[winner], '| moves:', int((board > 0).sum()))
plt.figure(figsize=(5, 5)); show(np.stack([board == 1, board == 2])); plt.show()